# 05_XGBoost_Model.ipynb
## FakeJobShield: XGBoost Hyperparameter Tuning
This notebook optimizes the XGBoost classifier using hyperparameter grid tuning over `max_depth`, `learning_rate`, `subsample`, and `n_estimators`. It then trains the best estimator, visualizes the feature importances, and exports the model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack, csr_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sns.set_theme(style="whitegrid")


In [ ]:
# Load cleaned dataset and reconstruct feature matrix
df = pd.read_csv("data/cleaned_fake_job_postings.csv")
df["fraudulent_int"] = df["fraudulent"].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0}).fillna(0).astype(int)
y = df["fraudulent_int"].values

# Vectorizer & Encoders
vectorizer = joblib.load("models/tfidf.pkl")
label_encoders = joblib.load("models/label_encoders.pkl")

# Features
x_tfidf = vectorizer.transform(df["cleaned_text"].fillna(""))
for col in ["telecommuting", "has_company_logo", "has_questions"]:
    df[col] = df[col].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0, True: 1, False: 0}).fillna(0).astype(int)
x_binary = df[["telecommuting", "has_company_logo", "has_questions"]].values

encoded_cats = []
for col in ["employment_type", "required_experience", "required_education", "industry", "function"]:
    df[col] = df[col].fillna("missing").astype(str)
    le = label_encoders[col]
    df[col + "_encoded"] = le.transform(df[col])
    encoded_cats.append(df[col + "_encoded"].values.reshape(-1, 1))
x_categorical = np.hstack(encoded_cats)

x_final = hstack([x_tfidf, csr_matrix(x_binary), csr_matrix(x_categorical)]).tocsr()
print("Final Combined Feature Matrix shape:", x_final.shape)


In [ ]:
# Split data
x_train, x_test, y_train, y_test = train_test_split(
    x_final, y, test_size=0.15, stratify=y, random_state=42
)


In [ ]:
# Setup XGBoost tuning grid
xgb = XGBClassifier(scale_pos_weight=20, eval_metric="logloss", random_state=42, n_jobs=-1)

param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.1, 0.2],
    'n_estimators': [100, 150]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Starting Grid Search...")
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(x_train, y_train)
print("Grid Search Complete!")
print("Best Parameters:", grid_search.best_params_)
print("Best F1 Score in CV:", grid_search.best_score_)


In [ ]:
# Evaluate Best XGBoost Model
best_xgb = grid_search.best_estimator_
test_preds = best_xgb.predict(x_test)
test_probs = best_xgb.predict_proba(x_test)[:, 1]

print("Test Performance Report:")
print(classification_report(y_test, test_preds))
print("Test F1 Score:", f1_score(y_test, test_preds))
print("Test ROC AUC Score:", roc_auc_score(y_test, test_probs))


In [ ]:
# Plot Feature Importance
feature_names = (
    list(vectorizer.get_feature_names_out()) + 
    ["telecommuting", "has_company_logo", "has_questions"] + 
    ["employment_type", "required_experience", "required_education", "industry", "function"]
)

importances = best_xgb.feature_importances_
indices = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette="viridis")
plt.title("Top 20 Feature Importances - Best XGBoost")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.savefig("results/xgboost_feature_importance.png", dpi=150)
plt.show()


In [ ]:
# Save Best Tuned XGBoost Model
joblib.dump(best_xgb, "models/best_xgboost.pkl")
print("Saved tuned XGBoost model to 'models/best_xgboost.pkl'")
